# Pareto Front: Configurable Cross-Campaign Comparison (WCM Basis)

This notebook compares all available simulation runs from selected campaigns on a **Wet Cell Matter (WCM)** basis using:

- Feed Conversion Efficiency (FCE, kg nutrient / kg WCM)
- Protein Conversion Efficiency (PCE, kg protein / kg protein)
- Medium Conversion Efficiency (MCE, m³ medium / kg WCM)
- Space-Time Yield (STY, kg WCM / m³ / day)

Campaigns are shown by color and operating modes by marker symbol. You can configure which campaigns to compare in the configuration cell below.

# Configuration

- **`SELECTED_CAMPAIGNS`**: List of campaign keys to compare (e.g. `["Campaign_1", "Campaign_6"]` or `["Campaign_1", "Campaign_2", "Campaign_3"]` etc.).
- **`EXPORT_FIGURES`**: Set to `True` to write SVG files for all Pareto figures, or `False` to display figures without exporting them.

In [16]:
# Configure which campaigns to compare. Available keys: "Campaign_1", "Campaign_2", "Campaign_3", "Campaign_4", "Campaign_5", "Campaign_6"
# Can be 2, 3, or more campaigns.
SELECTED_CAMPAIGNS = ["Campaign_4", "Campaign_5"]

# Set to True to export every displayed Pareto figure as SVG.
EXPORT_FIGURES = True

# Set to True to display conventional meat production benchmarks (Beef, Pork, Poultry)
SHOW_MEAT_BENCHMARKS = False

In [17]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import plotly.graph_objects as go


# Resolve paths from the workspace layout. Override ROOT manually if needed.
ROOT = Path.cwd().parent
if not (ROOT / "Campaign_1_fixed_stoichiometry_Humbird").exists():
    ROOT = Path(r"C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code")

OUTPUT_DIR = ROOT / "Cross_Campaign_Comparison" / "export"

# Registry of all available campaigns and their paths
AVAILABLE_CAMPAIGNS = {
    "Campaign_1": {
        "path": ROOT / "Campaign_1_fixed_stoichiometry_Humbird" / "06_Analysis" / "summary_comparison.csv",
        "display_name": "1. Humbird (2020) - Base Case",
        "color": "#56B4E9",
    },
    "Campaign_6": {
        "path": ROOT / "Campaign_6_fixed_stoichiometry_Humbird_enhanced" / "06_Analysis" / "summary_comparison.csv",
        "display_name": "2. Humbird (2020) - Enhanced",
        "color": "#E69F00",
    },
    "Campaign_2": {
        "path": ROOT / "Campaign_2_fixed_stoichiometry_GFI" / "06_Analysis" / "summary_comparison.csv",
        "display_name": "3. GFI (2020) - Fixed Stoich.",
        "color": "#0072B2",
    },
    # "Campaign_3": {
    #     "path": ROOT / "Campaign_3_dynamic_stoichiometry_GFI" / "06_Analysis" / "summary_comparison.csv",
    #     "display_name": "4. GFI (2020) - Dynamic Stoich.",
    #     "color": "#D55E00",
    # },
    "Campaign_4": {
        "path": ROOT / "Campaign_4_mode_comparison_excesskinetics" / "06_Analysis" / "summary_comparison.csv",
        "display_name": "4. Substrate Dependent",
        "color": "#009E73",
    },
    "Campaign_5": {
        "path": ROOT / "Campaign_5_mode_comparison_adjustedexcesskinetics" / "06_Analysis" / "summary_comparison.csv",
        "display_name": "5. Substrate Dependent - Enhanced",
        "color": "#CC79A7",
    },
}


# Conventional meat production benchmarks (raw literature values on wet weight basis)
MEAT_BENCHMARKS = {
    "Beef": {
        "STY_WCM_kg_m3_day": 0.1151,
        "medium_conversion_efficiency_m3_kg_WCM": 0.4111,
        "protein_conversion_efficiency_kg_kg": 28.80,
        "feed_conversion_efficiency_kg_kg_WCM": 36.00,
        "color": "#8B1E0F",
        "symbol": "square",
    },
    "Pork": {
        "STY_WCM_kg_m3_day": 0.3381,
        "medium_conversion_efficiency_m3_kg_WCM": 0.3148,
        "protein_conversion_efficiency_kg_kg": 7.29,
        "feed_conversion_efficiency_kg_kg_WCM": 6.00,
        "color": "#D95F02",
        "symbol": "diamond",
    },
    "Poultry": {
        "STY_WCM_kg_m3_day": 0.5398,
        "medium_conversion_efficiency_m3_kg_WCM": 0.3879,
        "protein_conversion_efficiency_kg_kg": 3.57,
        "feed_conversion_efficiency_kg_kg_WCM": 4.20,
        "color": "#2E7D32",
        "symbol": "triangle-up",
    },
}

# Validate and build active campaign mapping from SELECTED_CAMPAIGNS
CAMPAIGN_FILES = {}
CAMPAIGN_DISPLAY_NAMES = {}
CAMPAIGN_COLORS = {}

for campaign_key in SELECTED_CAMPAIGNS:
    if campaign_key not in AVAILABLE_CAMPAIGNS:
        raise ValueError(
            f"Unknown campaign '{campaign_key}'. Available options: {list(AVAILABLE_CAMPAIGNS.keys())}"
        )
    cfg = AVAILABLE_CAMPAIGNS[campaign_key]
    CAMPAIGN_FILES[campaign_key] = cfg["path"]
    CAMPAIGN_DISPLAY_NAMES[campaign_key] = cfg["display_name"]
    CAMPAIGN_COLORS[campaign_key] = cfg["color"]

# Derive comparison label and filename slug (e.g. "C1_vs_C6")
CAMPAIGN_LABEL = " vs ".join(
    CAMPAIGN_DISPLAY_NAMES[c].split(" — ")[0] for c in SELECTED_CAMPAIGNS
)
CAMPAIGN_SLUG = "_vs_".join(
    c.lower().replace("campaign_", "c") for c in SELECTED_CAMPAIGNS
)

MODE_SYMBOLS = {
    "Batch": "circle",
    "FedBatch": "square",
    "Chemostat": "diamond",
    "Concentrated FedBatch": "triangle-up",
    "Continuous Perfusion": "x",
}

MODE_DISPLAY_NAMES = {
    "Concentrated FedBatch": "FedBatch Perfusion",
}

MODE_ORDER = [
    "Batch",
    "FedBatch",
    "Chemostat",
    "Concentrated FedBatch",
    "Continuous Perfusion",
]

STY_WCM_COLUMN = "STY_WCM_kg_m3_day"

REQUIRED_COLUMNS = {
    "mode",
    "run_id",
    "feed_conversion_efficiency_kg_kg_WCM",
    STY_WCM_COLUMN,
}

# Lower conversion efficiency is preferable. Its reciprocal makes both
# Pareto objectives maximization objectives while preserving their ordering.
INVERSE_FEED_CONVERSION_COLUMN = "inverse_feed_conversion_efficiency_kg_kg_WCM"


def normalize_mode(value: object) -> str:
    mode = re.sub(r"^\d+_", "", str(value)).replace("_", " ")
    return {
        "Batch": "Batch",
        "FedBatch": "FedBatch",
        "Chemostat": "Chemostat",
        "ConcentratedFedBatch": "Concentrated FedBatch",
        "ContinuousPerfusion": "Continuous Perfusion",
    }.get(mode, mode)


def load_campaign_points() -> pd.DataFrame:
    frames = []
    for campaign, csv_path in CAMPAIGN_FILES.items():
        if not csv_path.exists():
            raise FileNotFoundError(f"Missing {campaign} input: {csv_path}")

        frame = pd.read_csv(csv_path)
        missing = REQUIRED_COLUMNS.difference(frame.columns)
        if missing:
            raise ValueError(f"{campaign} is missing required columns: {sorted(missing)}")

        frame = frame.copy()
        frame["campaign"] = campaign
        frame["mode_group"] = frame["mode"].map(normalize_mode)
        for column in [
            "feed_conversion_efficiency_kg_kg_WCM",
            "medium_conversion_efficiency_m3_kg_WCM",
            "protein_conversion_efficiency_kg_kg",
            STY_WCM_COLUMN,
        ]:
            if column in frame.columns:
                frame[column] = pd.to_numeric(frame[column], errors="coerce")
        frames.append(frame)

        print(f"Loaded {len(frame)} rows for {campaign}: {csv_path}")

    points = pd.concat(frames, ignore_index=True)
    points = points.replace([np.inf, -np.inf], np.nan).dropna(
        subset=["feed_conversion_efficiency_kg_kg_WCM", STY_WCM_COLUMN]
    )
    points = points[points["feed_conversion_efficiency_kg_kg_WCM"] > 0].copy()
    points[INVERSE_FEED_CONVERSION_COLUMN] = (
        1 / points["feed_conversion_efficiency_kg_kg_WCM"]
    )
    points["mode_group"] = pd.Categorical(
        points["mode_group"], categories=MODE_ORDER, ordered=True
    )
    print(f"Usable Pareto points: {len(points)}")
    print(points.groupby(["campaign", "mode_group"], observed=False).size())
    return points


def pareto_front_maximize(df: pd.DataFrame, x_col: str, y_col: str) -> pd.DataFrame:
    values = df[[x_col, y_col]].to_numpy(dtype=float)
    efficient = np.ones(len(values), dtype=bool)

    for index, (x_value, y_value) in enumerate(values):
        if not efficient[index]:
            continue
        dominates = (
            (values[:, 0] >= x_value)
            & (values[:, 1] >= y_value)
            & ((values[:, 0] > x_value) | (values[:, 1] > y_value))
        )
        if np.any(dominates):
            efficient[index] = False

    return (
        df.loc[efficient]
        .drop_duplicates(subset=[x_col, y_col])
        .sort_values([x_col, y_col])
        .reset_index(drop=True)
    )


def create_conversion_pareto_figure(
    points: pd.DataFrame,
    *,
    source_column: str,
    inverse_column: str,
    metric_name: str,
    abbreviation: str,
    inverse_label: str,
    x_column: str = STY_WCM_COLUMN,
    x_label: str = "STY<sub>WCM</sub> (kg<sub>WCM</sub> m<sup>-3</sup> day<sup>-1</sup>)",
    show_meat_benchmarks: bool = True,
) -> tuple[go.Figure, dict[str, pd.DataFrame]]:
    if source_column not in points.columns:
        raise ValueError(f"Missing conversion-efficiency column: {source_column}")

    plot_points = points.copy()
    plot_points[source_column] = pd.to_numeric(plot_points[source_column], errors="coerce")
    plot_points = plot_points.replace([np.inf, -np.inf], np.nan).dropna(
        subset=[source_column, x_column]
    )
    plot_points = plot_points[plot_points[source_column] > 0].copy()
    plot_points[inverse_column] = 1 / plot_points[source_column]

    figure = go.Figure()
    figure_fronts = {}

    for campaign in CAMPAIGN_FILES:
        campaign_points = plot_points[plot_points["campaign"] == campaign].copy()
        display_name = CAMPAIGN_DISPLAY_NAMES[campaign]
        color = CAMPAIGN_COLORS[campaign]

        for mode in MODE_ORDER:
            subset = campaign_points[campaign_points["mode_group"].astype(str) == mode]
            if subset.empty:
                continue

            customdata = np.column_stack(
                [
                    np.full(len(subset), display_name),
                    subset["mode_group"].astype(str).map(
                        lambda value: MODE_DISPLAY_NAMES.get(value, value)
                    ),
                    subset["run_id"].astype(str),
                    subset[source_column].to_numpy(),
                ]
            )
            figure.add_trace(
                go.Scatter(
                    x=subset[x_column],
                    y=subset[inverse_column],
                    mode="markers",
                    marker=dict(
                        size=14,
                        color=color,
                        symbol=MODE_SYMBOLS.get(mode, "circle"),
                        opacity=0.9,
                        line=dict(color="black", width=2),
                    ),
                    customdata=customdata,
                    hovertemplate=(
                        "<b>%{customdata[0]}</b><br>"
                        "Mode: %{customdata[1]}<br>"
                        "Run: %{customdata[2]}<br>"
                        f"{metric_name.capitalize()}: %{{customdata[3]:.2f}}<br>"
                        "STY<sub>WCM</sub>: %{x:.2f} kg/m³/day<br>"
                        f"1/{abbreviation}: %{{y:.2f}}<extra></extra>"
                    ),
                    showlegend=False,
                )
            )

        front = pareto_front_maximize(campaign_points, x_column, inverse_column)
        figure_fronts[campaign] = front
        print(f"{campaign} {metric_name} figure: {len(front)} Pareto-front point(s)")

        if not front.empty:
            figure.add_trace(
                go.Scatter(
                    x=front[x_column],
                    y=front[inverse_column],
                    mode="lines",
                    line=dict(color=color, width=4, dash="dot"),
                    hovertemplate=(
                        f"<b>{display_name} Pareto front</b><br>"
                        "STY<sub>WCM</sub>: %{x:.2f} kg/m³/day<br>"
                        f"{inverse_label}: %{{y:.2f}}<extra></extra>"
                    ),
                    showlegend=False,
                )
            )

    # Add conventional meat benchmarks if requested
    if show_meat_benchmarks:
        for meat_name, meat_cfg in MEAT_BENCHMARKS.items():
            sty_val = meat_cfg.get(x_column)
            raw_conv_val = meat_cfg.get(source_column)
            if sty_val is None or raw_conv_val is None or raw_conv_val <= 0:
                continue
            inv_conv_val = 1 / raw_conv_val

            figure.add_trace(
                go.Scatter(
                    x=[sty_val],
                    y=[inv_conv_val],
                    mode="markers+text",
                    marker=dict(
                        size=14,
                        color=meat_cfg["color"],
                        symbol=meat_cfg["symbol"],
                        line=dict(color="black", width=1.2),
                    ),
                    text=[f" {meat_name}"],
                    textposition="top right",
                    textfont=dict(size=18, color=meat_cfg["color"]),
                    name=f"{meat_name} benchmark",
                    legendgroup="meat_benchmarks",
                    hovertemplate=(
                        f"<b>Conventional {meat_name} Benchmark</b><br>"
                        + f"STY<sub>WCM</sub>: {sty_val:.2f} kg/m³/day<br>"
                        + f"{metric_name.capitalize()}: {raw_conv_val:.2f}<br>"
                        + f"1/{abbreviation}: {inv_conv_val:.2f}<extra></extra>"
                    ),
                    showlegend=True,
                )
            )

    for campaign in CAMPAIGN_FILES:
        figure.add_trace(
            go.Scatter(
                x=[None], y=[None], mode="markers",
                name=CAMPAIGN_DISPLAY_NAMES[campaign],
                marker=dict(
                    size=14,
                    color=CAMPAIGN_COLORS[campaign],
                    symbol="circle",
                    line=dict(color="black", width=2),
                ),
                hoverinfo="skip", showlegend=True,
            )
        )

    for mode in MODE_ORDER:
        if not (plot_points["mode_group"].astype(str) == mode).any():
            continue
        figure.add_trace(
            go.Scatter(
                x=[None], y=[None], mode="markers",
                name=MODE_DISPLAY_NAMES.get(mode, mode),
                marker=dict(size=14, symbol=MODE_SYMBOLS[mode], color="rgba(90,90,90,0.95)"),
                hoverinfo="skip", showlegend=True,
            )
        )

    figure.add_trace(
        go.Scatter(
            x=[None, None], y=[None, None], mode="lines", name="Pareto front",
            line=dict(color="rgba(80,80,80,0.95)", width=2, dash="dot"),
            hoverinfo="skip", showlegend=True,
        )
    )
    figure.update_layout(
        template="plotly_white",
        title=f"{CAMPAIGN_LABEL}: STY_WCM vs 1/{abbreviation}",
        width=1000,
        height=620,
        legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
        margin=dict(l=80, r=300, t=70, b=70),
    )
    figure.update_xaxes(
        title_text=x_label,
        title_font=dict(size=18),
        tickfont=dict(size=18),
        rangemode="tozero",
        showgrid=True, gridcolor="rgba(0,0,0,0.10)", zeroline=False,
    )
    figure.update_yaxes(
        title_text=inverse_label,
        title_font=dict(size=18),
        tickfont=dict(size=18),
        rangemode="tozero",
        showgrid=True, gridcolor="rgba(0,0,0,0.10)", zeroline=False,
    )

    return figure, figure_fronts


def export_figure(figure: go.Figure, output_path: Path) -> None:
    if not EXPORT_FIGURES:
        return

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    svg_figure = go.Figure(figure)
    svg_figure.update_layout(
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
    )
    svg_figure.write_image(str(output_path), format="svg", width=1000, height=620)
    print(f"Saved: {output_path}")


points_all = load_campaign_points()

Loaded 17 rows for Campaign_4: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_4_mode_comparison_excesskinetics\06_Analysis\summary_comparison.csv
Loaded 17 rows for Campaign_5: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_5_mode_comparison_adjustedexcesskinetics\06_Analysis\summary_comparison.csv
Usable Pareto points: 34
campaign    mode_group           
Campaign_4  Batch                    1
            FedBatch                 5
            Chemostat                1
            Concentrated FedBatch    5
            Continuous Perfusion     5
Campaign_5  Batch                    1
            FedBatch                 5
            Chemostat                1
            Concentrated FedBatch    5
            Continuous Perfusion     5
dtype: int64


In [18]:
# STY_WCM vs 1/FCE (WCM) Pareto Front

FCE_CONFIG = {
    "source_column": "feed_conversion_efficiency_kg_kg_WCM",
    "inverse_column": INVERSE_FEED_CONVERSION_COLUMN,
    "metric_name": "feed conversion efficiency (WCM)",
    "abbreviation": "FCE_WCM",
    "inverse_label": "1/FCE (kg<sub>WCM</sub> kg<sup>-1</sup>)",
    "show_meat_benchmarks": SHOW_MEAT_BENCHMARKS,
}
FCE_OUTPUT_PATH = OUTPUT_DIR / f"pareto_{CAMPAIGN_SLUG}_sty_wcm_vs_inverse_feed_conversion_wcm.svg"

fig_fce, fronts_fce = create_conversion_pareto_figure(points_all, **FCE_CONFIG)
fig_fce.show()
export_figure(fig_fce, FCE_OUTPUT_PATH)

Campaign_4 feed conversion efficiency (WCM) figure: 4 Pareto-front point(s)
Campaign_5 feed conversion efficiency (WCM) figure: 3 Pareto-front point(s)


Saved: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Cross_Campaign_Comparison\export\pareto_c4_vs_c5_sty_wcm_vs_inverse_feed_conversion_wcm.svg


In [19]:
# STY_WCM vs 1/PCE Pareto Front

PCE_CONFIG = {
    "source_column": "protein_conversion_efficiency_kg_kg",
    "inverse_column": "inverse_protein_conversion_efficiency_kg_kg",
    "metric_name": "protein conversion efficiency",
    "abbreviation": "PCE",
    "inverse_label": "1/PCE (kg<sub>protein</sub> kg<sup>-1</sup>)",
    "show_meat_benchmarks": SHOW_MEAT_BENCHMARKS,
}
PCE_OUTPUT_PATH = OUTPUT_DIR / f"pareto_{CAMPAIGN_SLUG}_sty_wcm_vs_inverse_protein_conversion.svg"

fig_pce, fronts_pce = create_conversion_pareto_figure(points_all, **PCE_CONFIG)
fig_pce.show()
export_figure(fig_pce, PCE_OUTPUT_PATH)

Campaign_4 protein conversion efficiency figure: 4 Pareto-front point(s)
Campaign_5 protein conversion efficiency figure: 4 Pareto-front point(s)


Saved: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Cross_Campaign_Comparison\export\pareto_c4_vs_c5_sty_wcm_vs_inverse_protein_conversion.svg


In [20]:
# STY_WCM vs 1/MCE (WCM) Pareto Front

MCE_CONFIG = {
    "source_column": "medium_conversion_efficiency_m3_kg_WCM",
    "inverse_column": "inverse_medium_conversion_efficiency_kg_m3_WCM",
    "metric_name": "medium conversion efficiency (WCM)",
    "abbreviation": "MCE_WCM",
    "inverse_label": "1/MCE (kg<sub>WCM</sub> m<sup>-3</sup>)",
    "show_meat_benchmarks": SHOW_MEAT_BENCHMARKS,
}
MCE_OUTPUT_PATH = OUTPUT_DIR / f"pareto_{CAMPAIGN_SLUG}_sty_wcm_vs_inverse_medium_conversion_wcm.svg"

fig_mce, fronts_mce = create_conversion_pareto_figure(points_all, **MCE_CONFIG)
fig_mce.show()
export_figure(fig_mce, MCE_OUTPUT_PATH)

Campaign_4 medium conversion efficiency (WCM) figure: 3 Pareto-front point(s)
Campaign_5 medium conversion efficiency (WCM) figure: 4 Pareto-front point(s)


Saved: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Cross_Campaign_Comparison\export\pareto_c4_vs_c5_sty_wcm_vs_inverse_medium_conversion_wcm.svg
